# Aula 07.1 Encriptação, Decriptação e Verificação de Integridade de Arquivos

**QXD0099 - Desenvolvimento de Software para Persistência**  
Universidade Federal do Ceará - Campus Quixadá  
Prof. Francisco Victor da Silva Pinheiro  
victorpinheiro@ufc.br

> Notebook prático e comentado para execução em sala no Google Colab.

## Agenda

### Criptografia
- O que é criptografia?
- Objetivos da criptografia
- Criptografia simétrica
- AES / Fernet
- Criptografia assimétrica
- RSA
- Comparação entre simétrica e assimétrica

### Integridade
- Hash
- MD5
- SHA-1
- SHA-256
- Checksum
- Ferramentas de terminal
- Verificação de integridade em Python

# 1. O que é criptografia?

Criptografia transforma informações legíveis em dados cifrados.

```text
Texto original
      ↓
   algoritmo
      +
     chave
      ↓
Texto cifrado
```

Para recuperar o conteúdo original, é necessário utilizar o mecanismo e a chave apropriados.

# 2. Objetivos da criptografia

A criptografia pode contribuir para:

- **Confidencialidade** — impedir leitura por pessoas não autorizadas;
- **Integridade** — detectar alterações;
- **Autenticação** — verificar identidade ou origem;
- **Não repúdio** — fornecer evidências de autoria em mecanismos como assinaturas digitais.

Criptografia e hash não são a mesma coisa.

- criptografia: geralmente permite recuperar o conteúdo original;
- hash: é uma transformação de mão única, usada principalmente para identificação e integridade.

# 3. Tipos de criptografia

## Simétrica

Utiliza a **mesma chave** para cifrar e decifrar.

```text
Arquivo
  ↓
Chave K
  ↓
Criptografia
  ↓
Arquivo cifrado
  ↓
Chave K
  ↓
Decriptação
  ↓
Arquivo original
```

## Assimétrica

Utiliza um **par de chaves**:

- chave pública;
- chave privada.

# 4. Preparando um arquivo para os exemplos

In [ ]:
from pathlib import Path

conteudo = (
    "QXD0099 - Desenvolvimento de Software para Persistência\n"
    "Universidade Federal do Ceará - Campus Quixadá\n"
    "Conteúdo confidencial para demonstração de criptografia.\n"
)

Path("arquivo.txt").write_text(
    conteudo,
    encoding="utf-8"
)

print(
    Path("arquivo.txt").read_text(
        encoding="utf-8"
    )
)

QXD0099 - Desenvolvimento de Software para Persistência
Universidade Federal do Ceará - Campus Quixadá
Conteúdo confidencial para demonstração de criptografia.



# 5. Criptografia simétrica

Para a prática usaremos a biblioteca `cryptography`.

O exemplo utiliza **Fernet**, uma construção de alto nível que já combina criptografia autenticada e gerenciamento de formato de mensagem.

In [ ]:
!pip -q install cryptography

## 6. Gerando uma chave simétrica

In [ ]:
from cryptography.fernet import Fernet

# Gera uma nova chave.
key = Fernet.generate_key()

print("Chave gerada:")
print(key)

Chave gerada:
b'S8sa6Pv0LKz5xWcvLiYYOOkn7kiYQJkkkM721vCQ0LU='


> Em sistemas reais, uma chave secreta não deve ser impressa em logs, versionada ou compartilhada publicamente.

## 7. Salvando a chave

In [ ]:
from pathlib import Path

Path("chave.key").write_bytes(
    key
)

print("Chave salva em chave.key")

Chave salva em chave.key


## 8. Criando o objeto de cifra

In [ ]:
from cryptography.fernet import Fernet

cipher_suite = Fernet(
    key
)

print("Objeto Fernet criado.")

Objeto Fernet criado.


## 9. Encriptando o arquivo

In [ ]:
from pathlib import Path

# Leitura em modo binário.
data = Path(
    "arquivo.txt"
).read_bytes()

# Cifra os bytes.
encrypted_data = cipher_suite.encrypt(
    data
)

Path(
    "arquivo_encrypted.bin"
).write_bytes(
    encrypted_data
)

print(
    "Arquivo cifrado criado:",
    "arquivo_encrypted.bin"
)

Arquivo cifrado criado: arquivo_encrypted.bin


## 10. Visualizando os bytes cifrados

In [ ]:
dados_cifrados = Path(
    "arquivo_encrypted.bin"
).read_bytes()

print(
    dados_cifrados[:200]
)

b'gAAAAABqmgymYBNU4pOkzBCr9615sQw2RzMdh7Mkiu8I8VWG_2R261__hLkVO6LlBUS3Jptoj0gSwR6ZwjmyDKn5lWJfcDdXtaUh_DnPQ-LHzPLbkOfYoWOt16u_XNILdGcKD9WXrDxttTKDcTa4S4QEV23qNgBAqFgzp-kZynbvhZ5vtrs3V0nBpQNiZw0QQIvR34PX'


Os dados cifrados não devem ser interpretados como texto legível.

## 11. Decriptando o arquivo

In [ ]:
encrypted_data = Path(
    "arquivo_encrypted.bin"
).read_bytes()

decrypted_data = cipher_suite.decrypt(
    encrypted_data
)

Path(
    "arquivo_decrypted.txt"
).write_bytes(
    decrypted_data
)

print(
    Path("arquivo_decrypted.txt").read_text(
        encoding="utf-8"
    )
)

QXD0099 - Desenvolvimento de Software para Persistência
Universidade Federal do Ceará - Campus Quixadá
Conteúdo confidencial para demonstração de criptografia.



## 12. Comparando original e decriptado

In [ ]:
original = Path(
    "arquivo.txt"
).read_bytes()

restaurado = Path(
    "arquivo_decrypted.txt"
).read_bytes()

print(
    "Conteúdo idêntico?",
    original == restaurado
)

Conteúdo idêntico? True


# 13. Tentativa de decriptação com chave incorreta

Na criptografia simétrica, a chave correta é essencial.

In [ ]:
from cryptography.fernet import Fernet, InvalidToken

chave_errada = Fernet.generate_key()
cipher_errado = Fernet(chave_errada)

try:

    cipher_errado.decrypt(
        encrypted_data
    )

    print(
        "Decriptação realizada."
    )

except InvalidToken:

    print(
        "Falha: chave incorreta "
        "ou conteúdo alterado."
    )

Falha: chave incorreta ou conteúdo alterado.


# 14. Aplicações da criptografia simétrica

- proteção de arquivos;
- backups;
- bancos de dados;
- armazenamento local;
- comunicação segura;
- grandes volumes de dados.

A principal vantagem é o desempenho.

O principal desafio é: **como compartilhar a chave com segurança?**

# 15. Criptografia assimétrica

Na criptografia assimétrica temos:

```text
Chave pública
     ↓
 pode ser compartilhada

Chave privada
     ↓
 deve permanecer secreta
```

No exemplo com RSA:

```text
Arquivo
  ↓
Chave pública
  ↓
Cifra
  ↓
Arquivo cifrado
  ↓
Chave privada
  ↓
Decifra
  ↓
Arquivo original
```

# 16. Gerando um par de chaves RSA

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa

private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

print(
    "Par de chaves RSA criado."
)

Par de chaves RSA criado.


## 17. Salvando a chave privada

In [ ]:
from cryptography.hazmat.primitives import serialization

private_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

Path(
    "private_key.pem"
).write_bytes(
    private_pem
)

print(
    "private_key.pem criado."
)

private_key.pem criado.


> Para fins didáticos, a chave privada está sendo salva sem senha. Em um sistema real, a chave privada deve receber proteção adequada.

## 18. Salvando a chave pública

In [ ]:
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

Path(
    "public_key.pem"
).write_bytes(
    public_pem
)

print(
    "public_key.pem criado."
)

public_key.pem criado.


## 19. Visualizando somente a chave pública

In [ ]:
print(
    Path(
        "public_key.pem"
    ).read_text(
        encoding="utf-8"
    )
)

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEA69gGvuM1hcEkl0eWXWH/
26ksVfyUGcrBRqiZRC10q/vfxyLXy6GsJJe/oDb1OZxoZQ++PDBlz3dS8VC9Akm5
+rYI0nA5b40kuUgSk2XadX6r/Hkbhjx927L/vMtIjWWv6SYA8oUn/M6KcLJC1Ivh
APMeR4hv+sZEKRhoUiVn1Gv37Id/FEZMTcRoH/93Ex7s1avOpgvqz5CIRGFGeb+1
p4biUa4aHCCh+xu8BN7TDuwsuDn77G+SLhdKMkdz5Gqv53Le2PiACm12ppZpTpOn
MAoPpvZHO/kkv4S3XeLOWHJe2z/a/cE9u1KeuWwJ7gAj7ZOGDerXJbXGJ9wWXHuf
LQIDAQAB
-----END PUBLIC KEY-----



# 20. Encriptando com RSA

Usaremos OAEP com SHA-256.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import hashes

# Para RSA direto, usamos uma mensagem pequena.
mensagem = (
    b"Arquivo confidencial QXD0099"
)

encrypted_rsa = public_key.encrypt(
    mensagem,
    padding.OAEP(
        mgf=padding.MGF1(
            algorithm=hashes.SHA256()
        ),
        algorithm=hashes.SHA256(),
        label=None
    )
)

Path(
    "mensagem_rsa_encrypted.bin"
).write_bytes(
    encrypted_rsa
)

print(
    "Mensagem cifrada com RSA."
)

Mensagem cifrada com RSA.


## Observação importante sobre RSA

RSA **não é normalmente usado para cifrar arquivos grandes diretamente**.

O uso típico é híbrido:

```text
Arquivo grande
   ↓
AES/Fernet
   ↓
dados cifrados

chave simétrica
   ↓
RSA
   ↓
chave simétrica cifrada
```

Assim combinamos:

- velocidade da criptografia simétrica;
- distribuição segura de chave com criptografia assimétrica.

## 21. Decriptando com RSA

In [ ]:
encrypted_rsa = Path(
    "mensagem_rsa_encrypted.bin"
).read_bytes()

decrypted_rsa = private_key.decrypt(
    encrypted_rsa,
    padding.OAEP(
        mgf=padding.MGF1(
            algorithm=hashes.SHA256()
        ),
        algorithm=hashes.SHA256(),
        label=None
    )
)

print(
    decrypted_rsa.decode(
        "utf-8"
    )
)

Arquivo confidencial QXD0099


# 22. Comparação: Simétrica x Assimétrica

| Aspecto | Simétrica | Assimétrica |
|---|---|---|
| Chaves | Uma | Pública + privada |
| Velocidade | Alta | Menor |
| Dados em massa | Excelente | Não ideal |
| Compartilhamento de chave | Desafio | Facilitado |
| Exemplo | AES/Fernet | RSA, ECC |

# 23. Introdução à integridade

Verificação de integridade busca responder:

> **O arquivo ainda possui exatamente o mesmo conteúdo?**

Um hash gera um resumo do conteúdo.

```text
Arquivo
  ↓
Hash
  ↓
e3b0c44298...
```

Se um byte mudar, o hash normalmente muda.

# 24. Principais algoritmos

| Algoritmo | Tamanho | Situação atual |
|---|---:|---|
| MD5 | 128 bits | Inseguro para usos criptográficos |
| SHA-1 | 160 bits | Inseguro para usos criptográficos |
| SHA-256 | 256 bits | Amplamente utilizado |
| Checksum simples | Variável | Útil para erros acidentais, não para segurança |

MD5 e SHA-1 ainda podem aparecer em sistemas legados ou em verificações não adversariais, mas não devem ser escolhidos para novos mecanismos de segurança.

# 25. MD5 com Python

In [ ]:
import hashlib

file_path = "arquivo.txt"

with open(
    file_path,
    "rb"
) as arquivo:

    file_data = arquivo.read()

md5_hash = hashlib.md5(
    file_data
).hexdigest()

print(
    "MD5:",
    md5_hash
)

MD5: ec436c78624ccfd86f0784c93f356e13


# 26. SHA-1 com Python

In [ ]:
with open(
    "arquivo.txt",
    "rb"
) as arquivo:

    dados = arquivo.read()

sha1_hash = hashlib.sha1(
    dados
).hexdigest()

print(
    "SHA-1:",
    sha1_hash
)

SHA-1: e8cc495acb3229040168e559386ae1c90c4bbf86


# 27. SHA-256 com Python

In [ ]:
with open(
    "arquivo.txt",
    "rb"
) as arquivo:

    dados = arquivo.read()

sha256_hash = hashlib.sha256(
    dados
).hexdigest()

print(
    "SHA-256:",
    sha256_hash
)

SHA-256: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9


# 28. Comparando os tamanhos dos hashes

In [ ]:
print(
    "MD5:",
    len(md5_hash),
    "caracteres hexadecimais"
)

print(
    "SHA-1:",
    len(sha1_hash),
    "caracteres hexadecimais"
)

print(
    "SHA-256:",
    len(sha256_hash),
    "caracteres hexadecimais"
)

MD5: 32 caracteres hexadecimais
SHA-1: 40 caracteres hexadecimais
SHA-256: 64 caracteres hexadecimais


# 29. Ferramentas de terminal

No Linux/macOS:

```bash
md5sum arquivo.txt
sha1sum arquivo.txt
sha256sum arquivo.txt
```

In [ ]:
!md5sum arquivo.txt
!sha1sum arquivo.txt
!sha256sum arquivo.txt

ec436c78624ccfd86f0784c93f356e13  arquivo.txt
e8cc495acb3229040168e559386ae1c90c4bbf86  arquivo.txt
6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9  arquivo.txt


# 30. Função genérica para calcular hash

Para arquivos maiores, é melhor ler em blocos.

In [ ]:
import hashlib

def calcular_hash(
    caminho,
    algoritmo="sha256",
    tamanho_bloco=8192
):

    hash_function = hashlib.new(
        algoritmo
    )

    with open(
        caminho,
        "rb"
    ) as arquivo:

        while True:

            bloco = arquivo.read(
                tamanho_bloco
            )

            if not bloco:
                break

            hash_function.update(
                bloco
            )

    return hash_function.hexdigest()

## 31. Usando a função

In [ ]:
print(
    "MD5:",
    calcular_hash(
        "arquivo.txt",
        "md5"
    )
)

print(
    "SHA-1:",
    calcular_hash(
        "arquivo.txt",
        "sha1"
    )
)

print(
    "SHA-256:",
    calcular_hash(
        "arquivo.txt",
        "sha256"
    )
)

MD5: ec436c78624ccfd86f0784c93f356e13
SHA-1: e8cc495acb3229040168e559386ae1c90c4bbf86
SHA-256: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9


# 32. Salvando um hash de referência

In [ ]:
hash_referencia = calcular_hash(
    "arquivo.txt",
    "sha256"
)

Path(
    "checksum.txt"
).write_text(
    hash_referencia,
    encoding="utf-8"
)

print(
    "Hash salvo:"
)

print(
    Path(
        "checksum.txt"
    ).read_text(
        encoding="utf-8"
    )
)

Hash salvo:
6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9


# 33. Verificando a integridade

In [ ]:
hash_esperado = Path(
    "checksum.txt"
).read_text(
    encoding="utf-8"
).strip()

hash_atual = calcular_hash(
    "arquivo.txt",
    "sha256"
)

print(
    "Esperado:",
    hash_esperado
)

print(
    "Atual:",
    hash_atual
)

if hash_atual == hash_esperado:

    print(
        "Integridade verificada."
    )

else:

    print(
        "Integridade comprometida."
    )

Esperado: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9
Atual: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9
Integridade verificada.


# 34. Alterando o arquivo

Vamos alterar apenas uma linha e verificar novamente.

In [ ]:
with open(
    "arquivo.txt",
    "a",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        "ALTERAÇÃO NO ARQUIVO\n"
    )

print(
    "Arquivo modificado."
)

Arquivo modificado.


## 35. Verificando novamente

In [ ]:
hash_atual = calcular_hash(
    "arquivo.txt",
    "sha256"
)

print(
    "Esperado:",
    hash_esperado
)

print(
    "Atual:",
    hash_atual
)

if hash_atual == hash_esperado:

    print(
        "Integridade verificada."
    )

else:

    print(
        "Integridade comprometida: "
        "o arquivo foi alterado."
    )

Esperado: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9
Atual: a3ec42533f3528a7363a5843001dcc1f2d69db9c00ea264dfca55b45d53c1b30
Integridade comprometida: o arquivo foi alterado.


# 36. Checksum simples

Um checksum simples não possui as mesmas propriedades criptográficas de SHA-256.

Vamos criar uma soma simples dos bytes apenas para demonstrar o conceito.

In [ ]:
def checksum_simples(
    caminho
):

    soma = 0

    with open(
        caminho,
        "rb"
    ) as arquivo:

        while True:

            bloco = arquivo.read(
                8192
            )

            if not bloco:
                break

            soma += sum(
                bloco
            )

    # Simulação de checksum de 32 bits.
    return soma % (2 ** 32)


print(
    "Checksum simples:",
    checksum_simples(
        "arquivo.txt"
    )
)

Checksum simples: 18228


Checksum simples é útil para detectar erros acidentais, mas não deve ser tratado como mecanismo de segurança contra adulteração intencional.

# 37. Restaurando o arquivo original

Para os próximos exemplos, vamos remover a alteração feita anteriormente.

In [ ]:
Path(
    "arquivo.txt"
).write_text(
    conteudo,
    encoding="utf-8"
)

print(
    "Arquivo original restaurado."
)

Arquivo original restaurado.


# 38. Mini laboratório — encriptar + verificar integridade

Fluxo:

```text
arquivo.txt
     ↓
SHA-256 original
     ↓
Fernet
     ↓
arquivo cifrado
     ↓
decriptação
     ↓
SHA-256 restaurado
     ↓
comparação
```

In [ ]:
from cryptography.fernet import Fernet

# 1. Hash do original.
hash_original = calcular_hash(
    "arquivo.txt",
    "sha256"
)

# 2. Nova chave simétrica.
chave_lab = Fernet.generate_key()
fernet_lab = Fernet(
    chave_lab
)

# 3. Encriptação.
dados = Path(
    "arquivo.txt"
).read_bytes()

cifrado = fernet_lab.encrypt(
    dados
)

Path(
    "arquivo_lab.enc"
).write_bytes(
    cifrado
)

# 4. Decriptação.
restaurado = fernet_lab.decrypt(
    Path(
        "arquivo_lab.enc"
    ).read_bytes()
)

Path(
    "arquivo_lab_restaurado.txt"
).write_bytes(
    restaurado
)

# 5. Novo hash.
hash_restaurado = calcular_hash(
    "arquivo_lab_restaurado.txt",
    "sha256"
)

print(
    "Hash original:  ",
    hash_original
)

print(
    "Hash restaurado:",
    hash_restaurado
)

print(
    "Integridade preservada?",
    hash_original == hash_restaurado
)

Hash original:   6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9
Hash restaurado: 6ca62d550c77a8dd22c7d9e3c1b9c6677cd266c1a1a03272208aa38cbb938fd9
Integridade preservada? True


# 39. Atividade prática — Cofre de Arquivos

Implemente um programa que:

```text
1 - Gerar chave
2 - Encriptar arquivo
3 - Decriptar arquivo
4 - Gerar SHA-256
5 - Verificar integridade
0 - Sair
```

Arquivos sugeridos:

```text
documento.txt
chave.key
documento.enc
documento_restaurado.txt
documento.sha256
```

# 40. Solução de referência

In [ ]:
from pathlib import Path
from cryptography.fernet import Fernet, InvalidToken
import hashlib


ARQUIVO = Path(
    "documento.txt"
)

CHAVE = Path(
    "chave.key"
)

ARQUIVO_CIFRADO = Path(
    "documento.enc"
)

ARQUIVO_RESTAURADO = Path(
    "documento_restaurado.txt"
)

HASH_FILE = Path(
    "documento.sha256"
)


if not ARQUIVO.exists():

    ARQUIVO.write_text(
        "Documento confidencial da disciplina QXD0099.\n",
        encoding="utf-8"
    )


def gerar_chave():

    chave = Fernet.generate_key()

    CHAVE.write_bytes(
        chave
    )

    print(
        "Chave criada."
    )


def carregar_fernet():

    if not CHAVE.exists():

        print(
            "Gere a chave primeiro."
        )

        return None

    return Fernet(
        CHAVE.read_bytes()
    )


def encriptar():

    fernet = carregar_fernet()

    if not fernet:
        return

    dados = ARQUIVO.read_bytes()

    cifrado = fernet.encrypt(
        dados
    )

    ARQUIVO_CIFRADO.write_bytes(
        cifrado
    )

    print(
        "Arquivo encriptado."
    )


def decriptar():

    fernet = carregar_fernet()

    if not fernet:
        return

    if not ARQUIVO_CIFRADO.exists():

        print(
            "Arquivo cifrado não existe."
        )

        return

    try:

        restaurado = fernet.decrypt(
            ARQUIVO_CIFRADO.read_bytes()
        )

        ARQUIVO_RESTAURADO.write_bytes(
            restaurado
        )

        print(
            "Arquivo decriptado."
        )

    except InvalidToken:

        print(
            "Falha na decriptação."
        )


def gerar_sha256():

    valor = calcular_hash(
        ARQUIVO,
        "sha256"
    )

    HASH_FILE.write_text(
        valor,
        encoding="utf-8"
    )

    print(
        "SHA-256 salvo:"
    )

    print(
        valor
    )


def verificar_integridade():

    if not HASH_FILE.exists():

        print(
            "Gere o hash de referência primeiro."
        )

        return

    esperado = HASH_FILE.read_text(
        encoding="utf-8"
    ).strip()

    atual = calcular_hash(
        ARQUIVO,
        "sha256"
    )

    print(
        "Esperado:",
        esperado
    )

    print(
        "Atual:",
        atual
    )

    if esperado == atual:

        print(
            "Integridade verificada."
        )

    else:

        print(
            "Arquivo alterado."
        )


while True:

    print("\n============================")
    print("       COFRE DE ARQUIVOS")
    print("============================")
    print("1 - Gerar chave")
    print("2 - Encriptar")
    print("3 - Decriptar")
    print("4 - Gerar SHA-256")
    print("5 - Verificar integridade")
    print("0 - Sair")

    opcao = input(
        "Opção: "
    )

    if opcao == "1":
        gerar_chave()

    elif opcao == "2":
        encriptar()

    elif opcao == "3":
        decriptar()

    elif opcao == "4":
        gerar_sha256()

    elif opcao == "5":
        verificar_integridade()

    elif opcao == "0":
        print(
            "Programa encerrado."
        )
        break

    else:
        print(
            "Opção inválida."
        )


       COFRE DE ARQUIVOS
1 - Gerar chave
2 - Encriptar
3 - Decriptar
4 - Gerar SHA-256
5 - Verificar integridade
0 - Sair


# Fechamento

```text
Segurança de Arquivos
        │
        ├── Confidencialidade
        │       ↓
        │   Criptografia
        │     ├── Simétrica
        │     └── Assimétrica
        │
        └── Integridade
                ↓
              Hash
             ├── MD5
             ├── SHA-1
             └── SHA-256
```

## Pontos principais

- criptografia simétrica utiliza uma chave compartilhada;
- criptografia assimétrica utiliza chave pública e privada;
- simétrica é mais adequada para grandes volumes de dados;
- RSA é normalmente utilizado em conjunto com criptografia simétrica;
- hashes não são criptografia reversível;
- MD5 e SHA-1 não devem ser usados para novos mecanismos de segurança;
- SHA-256 é amplamente utilizado para verificação de integridade;
- checksum simples detecta erros, mas não substitui hash criptográfico;
- comparar hashes permite identificar alterações em arquivos.